# High-Performance Computational Matrices & Numeric Precision Suite
**Engineer:** Rasha Quadri  
**Core Stack:** Julia (LinearAlgebra, SparseArrays, DataFrames)


## Module 1: High-Dimensional Scale Invariance (Jacobi Engine)

In [1]:
using LinearAlgebra
using SparseArrays

function build_system(N)
    MAT = sparse(Tridiagonal(fill(-1.0, N - 1),
            fill(3.0, N),
            fill(-1.0, N - 1)))

    RHS = [2.0; fill(1.0, N - 2); 2.0]

    return MAT, RHS
end


# Jacobi Method

function jacobi_sparse(A, b, tol=0.5e-6, maxiter=10^7)

    n = length(b)

    Dinv = 1.0 ./ diag(A)
    R = A - Diagonal(diag(A))

    x = zeros(n)


    for k in 1:maxiter
        x_new = Dinv .* (b - R*x)

        fwd_err = norm(x_new .- 1.0, Inf)

        if fwd_err < tol
            back_err = norm(b - A*x_new, Inf)
            return x_new, k, fwd_err, back_err
        end

        x = x_new
    end

    error("Jacobi did not converge")
end



A, b = build_system(100)

x, steps, fwd_err, back_err = jacobi_sparse(A, b)

println("n = 100")
println("Steps: ", steps)
println("Forward Error: ", fwd_err)
println("Backward Error: ", back_err)

A, b = build_system(100000)

x, steps, fwd_err, back_err = jacobi_sparse(A, b)

println("n = 100000")
println("Steps: ", steps)
println("Forward Error: ", fwd_err)
println("Backword Error: ", back_err)

println("Matrix size: ", size(A))
    



n = 100
Steps: 36
Forward Error: 4.578409922295634e-7
Backward Error: 4.5785176461254906e-7
n = 100000
Steps: 36
Forward Error: 4.578409922295634e-7
Backword Error: 4.5785176461254906e-7
Matrix size: (100000, 100000)


### Results

The Jacobi method took 36 iterations for both `n = 100` and `n = 100000`.  
At first this might seem surprising, but it makes sense: each iteration updates each variable by combining its neighbors and scaling by the diagonal.  
Because the diagonal is larger than the off-diagonal entries, the error in each step shrinks by roughly the same factor, no matter how large `n` is.  

This is why the number of iterations needed to reach a forward error below `0.5 × 10^-6` is basically the same for both small and large systems.  
The backward error is around `10^-7`, showing that the computed solutions are very accurate.

## Module 2: Parameter Boundary Fractures & Diagonal Dominance

In [2]:
function build_system(N)
    MAT = sparse(Tridiagonal(fill(1.0, N - 1),
            fill(2.0, N),
            fill(1.0, N - 1)))

    RHS = zeros(N)
    RHS[1] = 1.0
    RHS[end] = -1.0

    return MAT, RHS
end


# Jacobi Method

function jacobi_sparse(A, b, tol=0.5e-3, maxiter=10^7)

    n = length(b)

    Dinv = 1.0 ./ diag(A)
    R = A - Diagonal(diag(A))

    x = zeros(n)


    for k in 1:maxiter
        x_new = Dinv .* (b - R*x)

        fwd_err = norm(x_new .- (-1).^(0:n-1), Inf)

        if fwd_err < tol
            back_err = norm(b - A*x_new, Inf)
            return x_new, k, fwd_err, back_err
        end

        x = x_new
    end

    error("Jacobi did not converge")
end



A, b = build_system(100)

x, steps, fwd_err, back_err = jacobi_sparse(A, b)

println("n = 100")
println("Steps: ", steps)
println("Forward Error: ", fwd_err)
println("Backward Error: ", back_err)

n = 100
Steps: 16209
Forward Error: 0.0004998940639289184
Backward Error: 4.836152216469713e-7


### Results

For `n = 100`, the Jacobi method took 16,209 iterations to reach three correct decimal places. Even though each diagonal entry is 2 and each off-diagonal entry is 1, the **sum of the off-diagonal entries in the interior rows equals the diagonal** (1 + 1 = 2).

This means the system is **not strictly diagonally dominant**, so each iteration only reduces the error slightly. As a result, Jacobi converges very slowly when starting from zeros, which is why it took so many steps.

The first and last rows are slightly easier, but the interior rows dominate the iteration count.

The backward error `5 × 10^-7` shows that the solution is still very accurate.

## Module 3: Convergence Velocity Tracker (Jacobi vs. Gauss-Seidel)

In [3]:
using LinearAlgebra
using SparseArrays

function build_system(N)
    MAT = sparse(Tridiagonal(fill(-1.0, N - 1),
            fill(3.0, N),
            fill(-1.0, N - 1)))

    RHS = [2.0; fill(1.0, N - 2); 2.0]

    return MAT, RHS
end


# Jacobi Method

function jacobi_sparse(A, b, tol=0.5e-6, maxiter=10^7)

    n = length(b)

    Dinv = 1.0 ./ diag(A)
    R = A - Diagonal(diag(A))

    x = zeros(n)

    for k in 1:maxiter
        x_new = Dinv .* (b - R*x)

        fwd_err = norm(x_new .- 1.0, Inf)

        if fwd_err < tol
            back_err = norm(b - A*x_new, Inf)
            return x_new, k, fwd_err, back_err
        end

        x = x_new
    end

    error("Jacobi did not converge")
end



# Gauss-Seidel method

function gauss_seidel_sparse(A, b; tol=0.5e-6, maxiter=10^7)
    
    n = length(b)
    
    x = zeros(n)
    
    for k in 1:maxiter
        x_old = copy(x)
        for i in 1:n
            
            sum1 = i > 1 ? -1.0 * x[i-1] : 0.0
            sum2 = i < n ? -1.0 * x_old[i+1] : 0.0
            x[i] = (b[i] - sum1 - sum2)/3.0
        end
       
        fwd_err = norm(x .- ones(n), Inf)  
        
        if fwd_err < tol
            back_err = norm(b - A*x, Inf)
            return x, k, fwd_err, back_err
        end
    end
    
    error("Gauss-Seidel did not converge")
end


# Comparing


A, b = build_system(100)

println("=== Jacobi Method ===")
@time begin
    x_j, steps_j, fwd_err_j, back_err_j = jacobi_sparse(A, b)
end
println("Steps: ", steps_j, ", Forward Error: ", fwd_err_j, ", Backward Error: ", back_err_j)

println("\n=== Gauss-Seidel Method ===")
@time begin
    x_gs, steps_gs, fwd_err_gs, back_err_gs = gauss_seidel_sparse(A, b)
end
println("Steps: ", steps_gs, ", Forward Error: ", fwd_err_gs, ", Backward Error: ", back_err_gs)

=== Jacobi Method ===
  0.079208 seconds (45.05 k allocations: 2.958 MiB, 99.55% compilation time)
Steps: 36, Forward Error: 4.578409922295634e-7, Backward Error: 4.5785176461254906e-7

=== Gauss-Seidel Method ===
  0.139025 seconds (52.40 k allocations: 2.588 MiB, 99.82% compilation time)
Steps: 21, Forward Error: 4.76837158203125e-7, Backward Error: 4.77933892018001e-7


### Results 

**Comparison of the results:**

The Jacobi method required 36 iterations to reach a forward error below the tolerance, while the Gauss-Siedel only needed 21 iterations.

This shows that Gauss-Siedel converges faster because it immediately uses the most recently updated values within each iteration, whereas Jacobi only uses values from the previous iteration.

Both methods produce very accurate solutions, with forward and backward errors on the order of `10^-7`, confirming that the computed solutions are extremely close to the exact solution.

The measured runtime differences are small here due to the small system size and compilation overhead, but for larger systems, Gauss-Siedel would generally be more efficient.

## Module 4: Structural Matrix Validation (Cholesky Factorization)

In [4]:
using LinearAlgebra

function cholesky_fact(C::AbstractMatrix)

    n, m = size(C)
    
    if n != m
        return false, "Matrix is not square.", nothing
    end


    if !issymmetric(C)
       return false, "Matrix is not symmetric.", nothing
    end

    try
        U = cholesky(C).U
        return true, "Matrix is symmetric positive definite (SPD).", U
    catch
        return false, "Matrix is symmetric but not positive definite (not SPD).", nothing
    end
end



println("=== Test Case 1: SPD matrix ===")
A1 = [4.0 2.0; 2.0 3.0]
status, msg, U = cholesky_fact(A1)
println(msg)
if status
    println("Cholesky factor U =\n", U)
    println("Check: U'U - A1 = zero matrix?\n", U'U - A1)
end



println("\n=== Test Case 2: Symmetric but not SPD ===")
A2 = [0.0 1.0; 1.0 0.0]
status, msg, U = cholesky_fact(A2)
println(msg)

println("\n=== Test Case 3: Not symmetric ===")
A3 = [1.0 2.0; 3.0 4.0]
status, msg, U = cholesky_fact(A3)
println(msg)






    


=== Test Case 1: SPD matrix ===
Matrix is symmetric positive definite (SPD).
Cholesky factor U =
[2.0 1.0; 0.0 1.4142135623730951]
Check: U'U - A1 = zero matrix?
[0.0 0.0; 0.0 4.440892098500626e-16]

=== Test Case 2: Symmetric but not SPD ===
Matrix is symmetric but not positive definite (not SPD).

=== Test Case 3: Not symmetric ===
Matrix is not symmetric.


### Results

### 1. SPD Matrix ([4 2; 2 3])

This matrix is symmetric `(A=A')` and positice definite because all its eigenvalues are positive. The Cholesky factorization succeeds, producing an upper-triangular factor `U` such that `U'U = A`. The small numerical differences are due to floating-point rounding errors. This confirms that Cholesky factorization works correctly for symmetric positive definite matrices.

### 2. Symmetric but not SPD Matrix ([0 1; 1 0])

This matrix is symmetric `(A=A')` but **not positive definite** because one of its eigenvalues is negative. The function correctly detects that Cholesky factorization is not possible and gives and error message.

### 3. Non-symmetric Matrix ([1 2; 3 4])

This matrix is **not symmetric** `(A!=A')`.
Cholesky factorization requires symmetry, so the function outputs an error. Even if it were positive definite, the lack of symmetry prevents factorization.



## Module 5: Bit-Signal Integrity & Catastrophic Cancellation

In [5]:
using DataFrames, Printf


f_original(x) = (1 - (1 - x)^3)/x
f_reform(x) = 3 - 3*x + x^2


x_values = 10.0 .^ (-1:-1:-14)


correct_digits(true_val, approx_val) = floor(-log10(abs(true_val - approx_val)/abs(true_val)))


f_orig_values = f_original.(x_values)
f_ref_values = f_reform.(x_values)
digits_values = correct_digits.(f_ref_values, f_orig_values)


df = DataFrame(
    x = x_values,
    f_original = f_orig_values,
    f_reform = f_ref_values,
    correct_digits = digits_values
)


println("Results for f(x) = 1 - (1-x)^3 / x")
println("-----------------------------------------------------------")
println("|     x      |    f_original    |     f_reform    | correct_digits |")
println("|:----------:|:----------------:|:---------------:|:--------------:|")


for i in 1:length(x_values)
    @printf("| %10.1e | %16.10f | %15.10f | %14d |\n",
            x_values[i], f_orig_values[i], f_ref_values[i], digits_values[i])
end

Results for f(x) = 1 - (1-x)^3 / x
-----------------------------------------------------------
|     x      |    f_original    |     f_reform    | correct_digits |
|:----------:|:----------------:|:---------------:|:--------------:|
|    1.0e-01 |     2.7100000000 |    2.7100000000 |             15 |
|    1.0e-02 |     2.9701000000 |    2.9701000000 |             14 |
|    1.0e-03 |     2.9970010000 |    2.9970010000 |             15 |
|    1.0e-04 |     2.9997000100 |    2.9997000100 |             12 |
|    1.0e-05 |     2.9999700001 |    2.9999700001 |             11 |
|    1.0e-06 |     2.9999970002 |    2.9999970000 |             10 |
|    1.0e-07 |     2.9999996987 |    2.9999997000 |              9 |
|    1.0e-08 |     2.9999999818 |    2.9999999700 |              8 |
|    1.0e-09 |     2.9999999152 |    2.9999999970 |              7 |
|    1.0e-10 |     3.0000002482 |    2.9999999997 |              7 |
|    1.0e-11 |     3.0000002482 |    3.0000000000 |              7 |
|    1.0

### Results


- **Original expression (`f(x)`)** suffers from subtractive cancellation for very small `x`. This causes the computed values to lose accuracy as `x` decreases. For example, by `x = 10^-12`, `f(x)` already differs noticeably from the reformulated value.

- **Reformulated expression (`f_reform(x)`)** avoids subtracting nearly equal numbers, so it maintains high accuracy even for very small `x`.

- The **`correct_digits`** column shows how many digits of the original formula are correct compared to the reformulated version. The number of correct digits decreases as `x` gets smaller, illustrating the effect of numerical cancellation in the original formula.

- For `x >= 10^-6`, the original formula is still fairly accurate, but below that, accuracy rapidly deteriorates, confirming that reformulation is essential for stable computation.